# AI-Based Road Safety Monitoring and Accident Risk Analysis System
**Student:** Junaid Naeem  |  **Roll number:** SU92-BCSFM-F26-041  |  **University:** Superior University

## 1. Project introduction
This notebook is the readable walkthrough of ONE integrated project. It calls the same functions in `src/` that `main.py` and the Streamlit app use, and it reads/writes the same folders. **Before running:** place the datasets as described in `data/README.md`, then use *Kernel > Restart & Run All*. This file ships without saved outputs because it was not executed on real data.

## 2. Problem statement and objectives
**Problem:** road accidents cause serious harm; we want to understand patterns in accident records and practise computer-vision analysis of road/motorcycle images.

**Objectives:** (1) clean and explore real accident records; (2) predict whether an accident is *fatal/serious* or *slight* without target leakage; (3) process images and detect/match features; (4) classify images with a traditional method if the data allow; (5) report limitations honestly.

The accident and image datasets are **independent**: no image belongs to any accident record.

## 3. Dataset sources and descriptions
See `LICENSE_AND_DATA_SOURCES.md` for dataset names, links, licences, counts and limitations (to be completed after download). Row/column counts printed below come from the actual files.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
import config
from src import data_loader as dl, data_cleaning, eda, preprocessing
import main            # reuse the exact same stage functions as the command line
config.ensure_dirs()

def show(path, width=900):
    from IPython.display import Image, display
    p = Path(path)
    display(Image(filename=str(p), width=width)) if p.exists() else print("Missing:", p.name)

main.stage_check()

## 4. Data loading and inspection
The raw file is loaded and never modified.

In [ ]:
raw = dl.load_accidents()
print("Raw shape (rows, columns):", raw.shape)
display(raw.head())
display(data_cleaning.inspect(raw)["missing_share"])

## 5. Data cleaning and preprocessing
Each decision is logged as *problem found -> method -> why*. Imputation, scaling and encoding are NOT done here: they live inside the modelling pipeline and are fitted on training data only, which prevents leakage.

In [ ]:
main.stage_clean()                       # cleans, saves data/processed/accidents_clean.csv and the summary
summary = dl.read_json(config.CLEANING_SUMMARY)
display(pd.DataFrame(summary["steps"])[["problem", "method", "why"]])
print("Target counts:", summary["target_counts"], "| severe share:", summary["severe_share"])
display(pd.Series({k: v for k, v in summary["feature_types"].items() if k != "excluded"}))
print("Excluded columns:", summary["feature_types"]["excluded"])

## 6. Exploratory data analysis and interpretations
All findings are computed from the data. Correlation and association never prove causation.

In [ ]:
df = dl.load_clean_accidents()
findings, figs = eda.run_eda(df)
for f in findings: print("-", f)
display(pd.read_csv(config.METRICS_DIR / "descriptive_statistics.csv", index_col=0))

In [ ]:
for name in figs.values():
    show(config.FIG_DIR / name)

**How to read the charts:** the target-distribution chart shows class imbalance; hourly and category charts compare the *share of severe accidents* with the dashed overall rate; the heatmap shows linear correlation only. Write your own short interpretation of each chart here after viewing it.

## 7. Basic image processing
Load (OpenCV uses BGR), inspect, grayscale, Gaussian blur (noise), CLAHE (local contrast for dark/hazy road scenes).

In [ ]:
main.stage_images()
proc = dl.read_json(config.METRICS_DIR / "image_processing.json")
display(pd.DataFrame(proc["images"]).drop(columns=["comparison_figure"]))
for p in proc["images"][:2]:
    show(config.FIG_DIR / p["comparison_figure"], 1000)

## 8. Canny edge detection and ORB keypoints
Canny thresholds are derived from the median brightness. ORB gives 256-bit binary descriptors (32 bytes).

In [ ]:
main.stage_match()
feat = dl.read_json(config.METRICS_DIR / "feature_matching.json")
display(pd.DataFrame(feat["per_image"]))
if feat.get("warning"): print("WARNING:", feat["warning"])
show(config.FIG_DIR / "edges_and_keypoints.png", 1000)

## 9. ORB descriptor matching
`BFMatcher` with Hamming distance, two strategies: *cross-check* (mutual best match) and *Lowe ratio test*. Because some matches are still wrong, we also count matches consistent with a single geometric transform (RANSAC homography). Expect worse results under large viewpoint, scale or lighting change and blur.

In [ ]:
display(pd.DataFrame({m: feat[m] for m in ("crosscheck", "ratio")}))
show(config.MATCH_DIR / feat["ratio"]["figure"], 1100)
show(config.MATCH_DIR / feat["crosscheck"]["figure"], 1100)

## 10. Image classification (if supported by the data)
ORB descriptors -> Bag of Visual Words (k-means vocabulary learned on **training images only**) -> SVM / k-NN. The split is group-aware so near-duplicate copies of one source photo stay on one side. If the dataset has fewer than two classes with enough images, the stage is skipped and the reason is shown.

In [ ]:
main.stage_classify()
cls = dl.read_json(config.METRICS_DIR / "image_classification.json")
if cls["status"] == "done":
    display(pd.DataFrame({n: {k: v for k, v in m.items() if k in ("accuracy", "precision_macro", "recall_macro", "f1_macro")} for n, m in cls["models"].items()}).T)
    for f in sorted(config.FIG_DIR.glob("img_cm_*.png")): show(f, 450)
else:
    print("Skipped:", cls["reason"])

## 11. Accident machine learning models
Prediction problem: *is an accident Fatal/Serious (1) or Slight (0)?* using only information available when the accident is recorded. Casualty counts and police attendance are excluded (target leakage). Four models are compared inside leakage-safe pipelines.

In [ ]:
main.stage_model()
table = pd.read_csv(config.METRICS_DIR / "accident_model_comparison.csv")
display(table.round(3))

## 12. Evaluation metrics and model comparison
Primary metric: **F1 of the Fatal/Serious class** - accuracy is misleading when severe accidents are the minority. Compare `train_f1` with `test_f1` for over/underfitting.

In [ ]:
summ = dl.read_json(config.METRICS_DIR / "accident_models_summary.json")
print("Baseline (always predict majority class) accuracy:", round(summ["majority_class_baseline_accuracy"], 3))
print("Best model:", summ["best_model"]); print(summ["justification"])
show(config.FIG_DIR / "model_comparison.png", 800)
for f in sorted(config.FIG_DIR.glob("cm_*.png")): show(f, 400)
show(config.FIG_DIR / "rf_feature_importance.png", 700)

## 13. Integrated findings

In [ ]:
main.stage_report()
print("EDA findings:"); [print("-", f) for f in findings]
print("\nBest accident model:", summ["best_model"])
print("Image pair matches (ratio test):", feat["ratio"]["matches"], "| RANSAC inliers:", feat["ratio"]["ransac_inliers"])
print("Full report:", config.REPORT_DIR / "final_report.md")

## 14. Limitations
* Observational records: associations and feature importances are **not** causal.
* Predictions are educational and do not guarantee or measure real road safety; class weighting makes probabilities uncalibrated.
* Slight accidents are under-reported; no traffic-exposure data.
* Classical ORB features are sensitive to viewpoint, lighting and blur; ratio-test matches can still be wrong.
* The two datasets are unrelated; no helmet-violation or live-video detection is implemented.

## 15. Final conclusion
Write your conclusion here from the numbers printed above (best model and why, strongest EDA findings, matching/classification outcome, main limitations). Do not quote any figure that was not produced by your run.

## 16. References
* Rublee, E., Rabaud, V., Konolige, K., Bradski, G. (2011). ORB: an efficient alternative to SIFT or SURF. *ICCV*.
* Canny, J. (1986). A computational approach to edge detection. *IEEE TPAMI*, 8(6).
* Zuiderveld, K. (1994). Contrast limited adaptive histogram equalization. *Graphics Gems IV*.
* Lowe, D. G. (2004). Distinctive image features from scale-invariant keypoints. *IJCV*, 60(2).
* Csurka, G., et al. (2004). Visual categorization with bags of keypoints. *ECCV Workshop on Statistical Learning in Computer Vision*.
* Pedregosa, F., et al. (2011). Scikit-learn: machine learning in Python. *JMLR*, 12.
* Bradski, G. (2000). The OpenCV library. *Dr. Dobb's Journal of Software Tools*.
* Dataset sources: see `LICENSE_AND_DATA_SOURCES.md`.